# Probe 9: profile one Q10 step

Runs `scripts/profile_step.py` on the real config (batch 32, fp16) and
prints the top CUDA/CPU ops.

In [ ]:
import glob, os, shutil, subprocess, sys
def find_dataset(slug, probe):
    for root in glob.glob(f'/kaggle/input/datasets/*/{slug}') + [f'/kaggle/input/{slug}']:
        if os.path.exists(os.path.join(root, probe)): return root
    raise FileNotFoundError(slug)
SRC = find_dataset('kron-corpus-2b','train.bpe.bin')
CODE = find_dataset('kron-llm-code-v8','kronllm')
DATA = '/kaggle/working/data_prof'
os.makedirs(DATA, exist_ok=True)
for name in ('train.bpe.bin','val.bpe.bin','tokenizer.json','tokenizer_meta.json'):
    src = os.path.join(SRC, name); dst = os.path.join(DATA, name)
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        print('copying', name, flush=True); shutil.copyfile(src, dst)
print('data ready', flush=True)

In [ ]:
env = dict(os.environ, PYTHONPATH=CODE, PYTHONUNBUFFERED='1',
           CUDA_VISIBLE_DEVICES='0',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
cmd = [sys.executable,'-u', os.path.join(CODE,'scripts','profile_step.py'),
       '--data-dir', DATA, '--steps', '10', '--batch-size', '32']
log = '/tmp/prof.log'
with open(log,'w') as fh:
    r = subprocess.run(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT)
print('exit', r.returncode)
print(open(log).read()[-9000:])
shutil.rmtree(DATA, ignore_errors=True)